Testing Purposes

In [ ]:
# !pip install numpy gdal earthpy matplotlib torch torchvision torchaudio scikit-learn opencv-python tqdm

In [4]:
# Imports

import os
import random
import shutil

import numpy as np

from osgeo import gdal, gdal_array

import torch
import torch.optim as optim
import torch.nn as nn
from torchvision import transforms
from skimage.transform import resize

from sklearn.model_selection import train_test_split

import cv2 as cv

import tqdm

from pathlib import Path

import pandas as pd

In [5]:
# Data Splitting 
content_dir = Path("/kaggle/input/cloud-masking-dataset/content/train")
image_base_dir = os.path.join(content_dir, "data")
mask_base_dir = os.path.join(content_dir, "masks")

# Check if image directory exists
if not os.path.exists(image_base_dir):
    raise FileNotFoundError(f"Data Doesn't Exist: {image_base_dir}")

# List and sort files to maintain consistent ordering
image_paths = sorted([
    os.path.join(image_base_dir, fname)
    for fname in os.listdir(image_base_dir) if fname.endswith(".tif")
])

mask_paths = sorted([
    os.path.join(mask_base_dir, fname)
    for fname in os.listdir(mask_base_dir) if fname.endswith(".tif")
])

# Ensure matching counts
assert len(image_paths) == len(mask_paths), "Mismatch between image and mask counts"

# Optional: shuffle deterministically
combined = list(zip(image_paths, mask_paths))
random.seed(42)
random.shuffle(combined)
image_paths, mask_paths = zip(*combined)

# Split 60% train, 20% val, 20% test
X_temp, X_test, y_temp, y_test = train_test_split(image_paths, mask_paths, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42)

# Now you have:
print(f"Train: {len(X_train)} images")
print(f"Validation: {len(X_val)} images")
print(f"Test: {len(X_test)} images")

Train: 6343 images
Validation: 2115 images
Test: 2115 images


In [6]:
# util functions 
def convert_to_nparr(image_path):
    dataset = gdal.Open(str(image_path))

    if dataset is None:
        raise ValueError(f"Unable to open image at path: {image_path}")
    
    bands = []

    for i in range(dataset.RasterCount):
        band_array = dataset.GetRasterBand(i + 1).ReadAsArray()
        bands.append(band_array)

    bands = np.stack(bands, axis=0)  # Shape: (channels, height, width)
    return bands

def dice_coefficient(pred_mask: torch.Tensor, true_mask: torch.Tensor, eps=1e-6) -> float:
    intersection = (pred_mask * true_mask).sum()
    total_pixels = pred_mask.sum() + true_mask.sum()
    dice = (2.0 * intersection + eps) / (total_pixels + eps)
    return dice.item()


def copy_files(arr, dest_dir):
    """
    Copy files from a list of paths to a destination directory.
    """
    # Create destination directory if it doesn't exist
    os.makedirs(dest_dir, exist_ok=True)

    for file_path in arr:
        # Get the filename from the full path
        filename = os.path.basename(file_path)

        # Construct the destination file path
        dest_file = os.path.join(dest_dir, filename)

        # Check if file already exists at destination
        if os.path.exists(dest_file):
            continue

        try:
            # Copy the file
            shutil.copy2(file_path, dest_file)
            # print(f"Copied: {filename}")
        except FileNotFoundError:
            print(f"Source file not found: {file_path}")
        except PermissionError:
            print(f"Permission denied: Unable to copy {file_path}")

def rle_encode(mask):
    """
    Encodes a binary mask using Run-Length Encoding (RLE).    
    Args:
        mask (np.ndarray): 2D binary mask (0s and 1s).
    Returns:
        str: RLE-encoded string, or a single space " " if mask is all zeros.
    """
    if np.sum(mask) == 0:
        return " "  # As it seems that kaggle reject nulls. We'll handle cloud-free images with empty spaces.
    
    pixels = mask.flatten(order='F')  # Flatten in column-major order
    pixels = np.concatenate([[0], pixels, [0]])  # Add padding to detect transitions
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1  # Get transition indices
    runs[1::2] -= runs[::2]  # Compute run lengths
    runs[::2] -= 1  # Make it 0-indexed instead of 1-indexed

    return " ".join(map(str, runs))  # Convert to string format

def load_image(image_path):
    img = convert_to_nparr(image_path)
    img = torch.from_numpy(img).float()  
    img = img.unsqueeze(0)  
    return img

def predict_mask(image_path, model, device):
    img_tensor = load_image(image_path).to(device)
    
    pred_mask = model(img_tensor)
    
    pred_mask = (pred_mask > 0.5).float()

    mask_img = pred_mask.squeeze().cpu().numpy()
    mask_img = resize(mask_img, (256, 256), order=0, preserve_range=True, anti_aliasing=False).astype(np.uint8)

    return mask_img


In [ ]:
# ===================================== VISUALIZATION =====================================

# # Set a seed for reproducibility
# seed = 0
# random.seed(seed)

# # Select 25 random images
# random_images = random.sample(satellite_images, 1)



# for i, image_path in enumerate(random_images):
#     base_image = os.path.basename(image_path)
#     mask_path = os.path.join(mask_base_dir, base_image)

#     satellite_image = gdal.Open(image_path)
#     mask_image = gdal.Open(mask_path)

#     satellite_image = convert_to_nparr(satellite_image)
#     mask_image = convert_to_nparr(mask_image)

#     print(f"Image shape: {satellite_image.shape}")  # (bands, height, width)

#     # Also works with earthpy
#     ep.plot_rgb(satellite_image, rgb=(3, 2, 1), title=f"{image_path}")


In [7]:
# Data Loaders

class SatelliteDataset(torch.utils.data.Dataset):
    def __init__(self, image_paths, mask_paths, transform=None):
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        mask_path = self.mask_paths[idx]

        image = convert_to_nparr(image_path)
        mask = convert_to_nparr(mask_path)

        image = torch.from_numpy(image).float()
        mask = torch.from_numpy(mask).float()

        if self.transform:
            image = self.transform(image)
            mask = self.transform(mask)

        return image, mask

In [10]:
# Architecture 
class FireModule(nn.Module):
    def __init__(self, in_channels, squeeze_channels, expand1x1_channels, expand3x3_channels):
        super(FireModule, self).__init__()
        self.squeeze = nn.Conv2d(in_channels, squeeze_channels, kernel_size=1)
        self.squeeze_activation = nn.ReLU(inplace=True)

        self.expand1x1 = nn.Conv2d(squeeze_channels, expand1x1_channels, kernel_size=1)
        self.expand1x1_activation = nn.ReLU(inplace=True)

        self.expand3x3 = nn.Conv2d(squeeze_channels, expand3x3_channels, kernel_size=3, padding=1)
        self.expand3x3_activation = nn.ReLU(inplace=True)

    def forward(self, x):
        x = self.squeeze_activation(self.squeeze(x))
        return torch.cat([
            self.expand1x1_activation(self.expand1x1(x)),
            self.expand3x3_activation(self.expand3x3(x))
        ], 1)


class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, dropout=False, batchnorm=True, dropout_prob=0.5):
        super(ConvBlock, self).__init__()
        layers = [
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels) if batchnorm else nn.Identity(),
            nn.ReLU(inplace=True),
        ]
        layers += [
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels) if batchnorm else nn.Identity(),
            nn.ReLU(inplace=True),
        ]
        if dropout:
            layers.append(nn.Dropout(dropout_prob))
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)


class UNet(nn.Module):
    in_channels = 4

    def prepare_image(self, x):
        return x

    def __init__(self, out_channels, dropout_prob=0.5):
        super(UNet, self).__init__()

        # Replace first encoding layer with SqueezeNet-like block
        self.enc1 = nn.Sequential(
            FireModule(self.in_channels, 8, 16, 16),  # Output: 32 channels (16+16)
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True)
        )
        self.pool1 = nn.MaxPool2d(2)  # 256x256

        self.enc2 = ConvBlock(32, 64)
        self.pool2 = nn.MaxPool2d(2)  # 128x128

        self.enc3 = ConvBlock(64, 128)
        self.pool3 = nn.MaxPool2d(2)  # 64x64

        self.enc4 = ConvBlock(128, 256)
        self.pool4 = nn.MaxPool2d(2)  # 32x32

        self.bottleneck = ConvBlock(256, 512)

        self.upconv4 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.dec4 = ConvBlock(512, 256, batchnorm=False)

        self.upconv3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec3 = ConvBlock(256, 128, batchnorm=False)

        self.upconv2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec2 = ConvBlock(128, 64, batchnorm=False)

        self.upconv1 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        self.dec1 = ConvBlock(64, 32, dropout=True, batchnorm=False, dropout_prob=dropout_prob)

        self.out_conv = nn.Conv2d(32, out_channels, kernel_size=1)

        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.prepare_image(x)

        enc1 = self.enc1(x)
        enc1_pool = self.pool1(enc1)

        enc2 = self.enc2(enc1_pool)
        enc2_pool = self.pool2(enc2)

        enc3 = self.enc3(enc2_pool)
        enc3_pool = self.pool3(enc3)

        enc4 = self.enc4(enc3_pool)
        enc4_pool = self.pool4(enc4)

        bottleneck = self.bottleneck(enc4_pool)

        dec4 = self.upconv4(bottleneck)
        dec4 = torch.cat((dec4, enc4), dim=1)
        dec4 = self.dec4(dec4)

        dec3 = self.upconv3(dec4)
        dec3 = torch.cat((dec3, enc3), dim=1)
        dec3 = self.dec3(dec3)

        dec2 = self.upconv2(dec3)
        dec2 = torch.cat((dec2, enc2), dim=1)
        dec2 = self.dec2(dec2)

        dec1 = self.upconv1(dec2)
        dec1 = torch.cat((dec1, enc1), dim=1)
        dec1 = self.dec1(dec1)

        dec0 = self.out_conv(dec1)
        out = self.sigmoid(dec0)
        return out

In [11]:
# Hyperparameters

BATCH_SIZE = 16
DROPOUT_PROB = 0.5
LEARNING_RATE = 0.001
NUM_EPOCHS = 20
CRITERION = nn.BCELoss()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = UNet(out_channels=1, dropout_prob=DROPOUT_PROB).to(device)  # 1 output channel for binary mask

OPTIMIZER = optim.SGD(model.parameters(), lr=LEARNING_RATE)

In [12]:
# Data Loaders


# Define the transform for training images
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(degrees=45),
])

train_dataset = SatelliteDataset(X_train, y_train, transform=train_transform)
val_dataset = SatelliteDataset(X_val, y_val)
test_dataset = SatelliteDataset(X_test, y_test)

train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = torch.utils.data.DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = torch.utils.data.DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False)



criterion = CRITERION
optimizer = OPTIMIZER

In [ ]:
# Training Loop
best_val_loss = float('-inf') 

for epoch in range(NUM_EPOCHS):
    model.train()
    train_loss = 0.0

    for img, mask in tqdm.tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}"):
        img = img.to(device)
        mask = mask.to(device)

        # Forward pass
        outputs = model(img)
        loss = criterion(outputs, mask)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    # Validation
    model.eval()
    total_dice = 0.0
    with torch.no_grad():
        for val_image, val_mask in val_loader:
            val_image, val_mask = val_image.to(device), val_mask.to(device)
            pred_mask = model(val_image)

            pred_mask = (pred_mask > 0.5).float()
            val_mask = (val_mask > 0.5).float()

            total_dice += dice_coefficient(pred_mask, val_mask)

    val_loss = total_dice / len(val_loader)

    print(
        f"Epoch [{epoch+1}/{NUM_EPOCHS}] - Train Loss: {train_loss:.4f} - Val Acc: {val_loss:.4f}"
    )

    # Save the best model
    if val_loss > best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "best_model.pkl")
        print(f"Best model saved at epoch {epoch+1} with Val Acc: {val_loss:.4f}")


0.5


Epoch 1/20: 100%|██████████| 397/397 [14:22<00:00,  2.17s/it]


Epoch [1/20] - Train Loss: 0.6924 - Val Acc: 0.8027
Best model saved at epoch 1 with Val Acc: 0.8027


Epoch 2/20: 100%|██████████| 397/397 [10:45<00:00,  1.63s/it]


Epoch [2/20] - Train Loss: 0.6883 - Val Acc: 0.8195
Best model saved at epoch 2 with Val Acc: 0.8195


...

Epoch [18/20] - Train Loss: 0.5907 - Val Acc: 0.8450


Epoch 19/20: 100%|██████████| 397/397 [11:28<00:00,  1.73s/it]


Epoch [19/20] - Train Loss: 0.5877 - Val Acc: 0.8562
Best model saved at epoch 19 with Val Acc: 0.8562


Epoch 20/20: 100%|██████████| 397/397 [10:44<00:00,  1.62s/it]


Epoch [20/20] - Train Loss: 0.5858 - Val Acc: 0.8526


In [ ]:
# Testing Loop

model = UNet(out_channels=1)

model.load_state_dict(torch.load('/kaggle/working/best_model.pkl', weights_only=True))
    
model.eval() 
model.to("cuda")

total_dice = 0.0

with torch.no_grad():
    for test_images, test_masks in test_loader:
        test_images, test_masks = test_images.to(device), test_masks.to(device)

        pred_masks = model(test_images)

        pred_masks = (pred_masks > 0.5).float()
        test_masks = (test_masks > 0.5).float()

        total_dice += dice_coefficient(pred_masks, test_masks)

    test_dice = total_dice / len(test_loader)
    print(f"Test Dice Coefficient: {test_dice:.4f}")

/tmp/ipykernel_31/218964279.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('/kaggle/working/best_model.pkl'))


Test Dice Coefficient: 0.8473


In [25]:
# Generating Submission.csv

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load model
model = UNet(out_channels=1)
model.load_state_dict(torch.load('/kaggle/input/cloudmaskingcode/best_model.pkl', weights_only=True))
model.eval()
model.to("cuda")

test_dir = Path("/kaggle/input/cloud-masking-test-set-satellite-cmp25-course/test/test/data")
csv_path = Path("/kaggle/input/cloud-masking-test-set-satellite-cmp25-course/sample_submission.csv")

# Load sample submission
df = pd.read_csv(csv_path, dtype={"id": str})
df["segmentation"] = ""  

for idx, row in df.iterrows():
    image_id = row["id"]
    image_path = os.path.join(test_dir, f"{image_id}.tif")

    print(f"⚫ processing {image_path}")
    pred_mask = predict_mask(image_path, model, device)

    # Encode predicted mask (resize to original if needed)
    encoded_mask = rle_encode(pred_mask)
    df.at[idx, "segmentation"] = encoded_mask

# Save results
df["id"] = df["id"].astype(str)
df.to_csv("submission.csv", index=False)
print("Submission file created: submission.csv")

Using device: cuda
⚫ processing /kaggle/input/cloud-masking-test-set-satellite-cmp25-course/test/test/data/402143.tif
⚫ processing /kaggle/input/cloud-masking-test-set-satellite-cmp25-course/test/test/data/877548.tif
⚫ processing /kaggle/input/cloud-masking-test-set-satellite-cmp25-course/test/test/data/942725.tif
⚫ processing /kaggle/input/cloud-masking-test-set-satellite-cmp25-course/test/test/data/769836.tif
⚫ processing /kaggle/input/cloud-masking-test-set-satellite-cmp25-course/test/test/data/263633.tif
⚫ processing /kaggle/input/cloud-masking-test-set-satellite-cmp25-course/test/test/data/747518.tif
⚫ processing /kaggle/input/cloud-masking-test-set-satellite-cmp25-course/test/test/data/632056.tif
⚫ processing /kaggle/input/cloud-masking-test-set-satellite-cmp25-course/test/test/data/331069.tif
⚫ processing /kaggle/input/cloud-masking-test-set-satellite-cmp25-course/test/test/data/104502.tif
⚫ processing /kaggle/input/cloud-masking-test-set-satellite-cmp25-course/test/test/data/00

In [19]:
!cp /kaggle/input/cloudmaskingcode/evaluate_pref.py /kaggle/working

In [ ]:
# Analyzing Model 
from evaluate_pref import profile

# Load the model
model = UNet(out_channels=1)
model.load_state_dict(torch.load('model_state_dict.pth'))

num_ops, num_params = profile(model, (16, 4, 512, 512))

print(f"Number of operations: {num_ops}")
print(f"Number of parameters: {num_params}")